# FiLM-Conditioned Attention-MIL — NSCLC Immune Gene Prediction
**Paper:** *FiLM-Conditioned Attention-Based MIL Reveals Subtype-Specific Morphological Encoding of Antigen Presentation and T-Cell Inflammation in NSCLC*

### Before running — checklist
1. GPU accelerator enabled: **Settings > Accelerator > GPU T4 x2**
2. Your h5 feature dataset attached: **+ Add Data > Your Datasets**
3. Your metadata CSV dataset attached: **+ Add Data > Your Datasets**
4. Internet enabled (Settings > Internet > On) — needed for GitHub clone
5. Update the **PATHS** cell below to match your Kaggle dataset slugs

## Setup

In [1]:
# NOTE: EDIT THESE PATHS TO MATCH YOUR KAGGLE DATASET NAMES
# Find your dataset slug at kaggle.com/datasets/YOUR_USERNAME/DATASET_NAME

LUAD_FEATURES = "/kaggle/input/datasets/lucashuitema/tcga-luad"
LUSC_FEATURES = "/kaggle/input/datasets/lucashuitema/tcga-lusc"
METADATA_CSV  = "/kaggle/input/datasets/lucashuitema/tcga-csv/tcga-nsclc-metadata.csv"

OUTPUT_DIR    = "/kaggle/working/results"
GITHUB_REPO   = "https://github.com/LHu1t/nsclc-immune-film-mil.git"

N_FOLDS     = 5
MAX_EPOCHS  = 50
PATIENCE    = 3

FILM_ENABLED = False

print("Paths configured. Proceed to next cell.")

Paths configured. Proceed to next cell.


### Install missing dependancies and check GPU

In [2]:
# Note: torch, numpy, pandas already pre-installed on Kaggle
import subprocess
subprocess.run(["pip", "install", "h5py", "scikit-learn", "scipy", "lifelines", "-q"], check=True)
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.3 MB/s eta 0:00:00
Dependencies installed.


In [3]:
# Verify GPU is available
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}",
              f"| VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found. Enable GPU: Settings > Accelerator > GPU T4 x2")

PyTorch version: 2.10.0+cu128
CUDA available:  True
  GPU 0: Tesla T4 | VRAM: 15.6 GB
  GPU 1: Tesla T4 | VRAM: 15.6 GB


### Clone nsclc-immune-film-mil GitHub Repo into Kaggle

In [4]:
# Clone your GitHub repo to get the latest training script
import os
import sys

REPO_DIR = "/kaggle/working/nsclc-immune-film-mil"

if os.path.exists(REPO_DIR):
    # Pull latest changes if already cloned
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout)
else:
    result = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("Clone failed:", result.stderr)
        raise RuntimeError("GitHub clone failed. Check GITHUB_REPO path and that Internet is ON.")

# Add src/ to Python path so we can import train_film_mil
src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Files in src:")
print(os.listdir(src_path))


Files in src:
['compare_film_vs_nofilm.py', 'build_metadata.py', 'train_film_mil.py', 'make_paper_figures.py']


### Check filepaths and retrieve checkpoints

In [5]:
# Verify data paths before starting expensive training
from pathlib import Path
import pandas as pd

errors = []

# Check feature directories
for name, path in [("LUAD features", LUAD_FEATURES), ("LUSC features", LUSC_FEATURES)]:
    p = Path(path)
    if not p.exists():
        errors.append(f"{name} not found: {path}")
    else:
        h5_files = list(p.glob("*.h5"))
        print(f" {name}: {len(h5_files)} .h5 files found")
        print(f"  Example: {h5_files[0].name if h5_files else 'NONE'}")

# Check metadata CSV
if not Path(METADATA_CSV).exists():
    errors.append(f"Metadata CSV not found: {METADATA_CSV}")
else:
    df_check = pd.read_csv(METADATA_CSV, nrows=3)
    print(f"\n Metadata CSV: {pd.read_csv(METADATA_CSV).shape[0]} rows, "
          f"{len(df_check.columns)} columns")
    fpkm_cols = [c for c in pd.read_csv(METADATA_CSV, nrows=0).columns
                 if c.endswith("_fpkm_uq") or c in ("TMB", "APM", "TIS")]
    print(f"Gene/target columns found: {len(fpkm_cols)}")

if errors:
    for e in errors:
        print(e)
    raise RuntimeError("Fix the paths above before continuing.")

print("\n All data paths verified. Ready to train.")

 LUAD features: 531 .h5 files found
  Example: TCGA-49-6745-01Z-00-DX1.6bf8a38c-5e3a-4b32-8870-03bc06c0db80.h5
 LUSC features: 512 .h5 files found
  Example: TCGA-46-3765-01Z-00-DX1.f45e4e30-e60c-40e5-a0b7-4513c0c37fda.h5

 Metadata CSV: 1017 rows, 48 columns
Gene/target columns found: 38

 All data paths verified. Ready to train.


In [6]:
# Checkpoint utilities
import json
import numpy as np

CHECKPOINT_FILE = "/kaggle/working/checkpoint.json"

def _json_default(obj):
    """Convert numpy scalar/array types to native Python types for json.dump."""
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def save_checkpoint(fold_results: list, completed_fold: int):
    """Save progress after each fold completes."""
    checkpoint = {
        "completed_fold": completed_fold,
        "fold_results":   fold_results,
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f, indent=2, default=_json_default)
    print(f"Checkpoint saved after fold {completed_fold}")

def load_checkpoint():
    """Load checkpoint if it exists, returns (fold_results, start_fold)."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE) as f:
            cp = json.load(f)
        start_fold = cp["completed_fold"] + 1
        print(f"Resuming from fold {start_fold} "
              f"({cp['completed_fold']} folds already completed)")
        return cp["fold_results"], start_fold
    return [], 0

print("Checkpoint utilities ready.")

Checkpoint utilities ready.


## Main Training

In [7]:
# Import training components
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from pathlib import Path

from train_film_mil import (
    load_metadata,
    FiLMDataset,
    FiLMMILModel,
    CompositeLoss,
    run_epoch,
    compute_gene_pccs,
    compute_panel_pcc,
    compute_auc,
    APM_GENES,
    TIS_GENES,
)

print("All imports successful.")

All imports successful.


In [8]:
# Load and preprocess metadata
df, gene_cols, clinical_cols, y_means, y_stds, panel_idx = load_metadata(METADATA_CSV)

gene_symbol_to_idx = {}
for i, g in enumerate(gene_cols):
    symbol = g.replace("_fpkm_uq", "")
    gene_symbol_to_idx[symbol] = i

# Build gene symbol > index map
gene_symbol_to_idx = {}
for i, g in enumerate(gene_cols):
    symbol = g.replace("_fpkm_uq", "")
    gene_symbol_to_idx[symbol] = i

panel_gene_fallback_idx = {
    "APM": [
        gene_symbol_to_idx[g]
        for g in APM_GENES
        if g in gene_symbol_to_idx
    ],
    "TIS": [
        gene_symbol_to_idx[g]
        for g in TIS_GENES
        if g in gene_symbol_to_idx
    ],
}

feature_dirs = {"LUAD": LUAD_FEATURES, "LUSC": LUSC_FEATURES}

# Fixed 80/20 train-dev / test split
rng       = np.random.default_rng(94)
all_sids  = df["submitter_id"].unique()
test_sids = set(rng.choice(all_sids, size=int(0.2 * len(all_sids)), replace=False))
dev_sids  = [s for s in all_sids if s not in test_sids]

df_test = df[df["submitter_id"].isin(test_sids)].reset_index(drop=True)
df_dev  = df[df["submitter_id"].isin(dev_sids)].reset_index(drop=True)

print(f"Total samples : {len(df)}")
print(f"Dev set       : {len(df_dev)}")
print(f"Test set      : {len(df_test)}")
print(f"Gene targets  : {len(gene_cols)}")
#print(f"Subtypes      : LUAD={( df['cancer_type']=='LUAD').sum()}, LUSC={(df['cancer_type']=='LUSC').sum()}")

Total samples : 987
Dev set       : 790
Test set      : 197
Gene targets  : 38


In [9]:
# Main training loop with per-fold checkpointing
import os

print(f"Folds:{N_FOLDS}, Patience: {PATIENCE}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=90)

# Resume from checkpoint if available
fold_results, start_fold = load_checkpoint()

for fold, (train_idx, val_idx) in enumerate(kf.split(df_dev)):

    # Skip already-completed folds
    if fold < start_fold:
        print(f"Skipping fold {fold} (already completed)")
        continue

    print(f"\n{'='*60}")
    print(f"FOLD {fold} / {N_FOLDS - 1}")
    print(f"{'='*60}")

    df_train = df_dev.iloc[train_idx].reset_index(drop=True)
    df_val   = df_dev.iloc[val_idx].reset_index(drop=True)

    # Datasets
    train_ds = FiLMDataset(df_train, feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=False)
    val_ds   = FiLMDataset(df_val,   feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=True)
    test_ds  = FiLMDataset(df_test,  feature_dirs, gene_cols, clinical_cols,
                           n_tiles=None, deterministic=True)

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2,)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2,)
    test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2,)

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

    # Model, loss, optimiser
    model     = FiLMMILModel(feat_dim=1536, n_genes=len(gene_cols), use_film=FILM_ENABLED).to(device)
    loss_fn   = CompositeLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=2, factor=0.5
    )

    best_val_pcc = -np.inf
    best_weights = None
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_pcc, _, _, _, _ = run_epoch(model, train_loader, loss_fn, optimizer, epoch, device, training=True, panel_idx=panel_idx, panel_gene_fallback_idx=panel_gene_fallback_idx,)
        val_loss, val_pcc, _, _, _, _ = run_epoch(model, val_loader, loss_fn, optimizer, epoch, device, training=False, panel_idx=panel_idx, panel_gene_fallback_idx=panel_gene_fallback_idx,)
        scheduler.step(val_loss)

        print(f"  Epoch {epoch:3d} | "
              f"Train loss={train_loss:.4f} PCC={train_pcc:.4f} | "
              f"Val loss={val_loss:.4f} PCC={val_pcc:.4f}")

        if val_pcc > best_val_pcc:
            best_val_pcc = val_pcc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            # Save best weights to disk immediately in case of disconnect
            torch.save(best_weights, f"{OUTPUT_DIR}/fold{fold}_best_model.pt")
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    # Test set evaluation
    model.load_state_dict(best_weights)
    model.to(device)
    _, _, test_preds, test_labels, test_panel_preds, test_panel_labels = run_epoch(model, test_loader, loss_fn, optimizer, 999, device, training=False, panel_idx=panel_idx, panel_gene_fallback_idx=panel_gene_fallback_idx,)

    gene_pccs = compute_gene_pccs(test_preds, test_labels)
    apm_pcc   = compute_panel_pcc(test_preds, test_labels, gene_cols, APM_GENES, gene_symbol_to_idx)
    tis_pcc   = compute_panel_pcc(test_preds, test_labels, gene_cols, TIS_GENES, gene_symbol_to_idx)
    apm_auc   = compute_auc(test_preds, test_labels, gene_cols, APM_GENES, gene_symbol_to_idx)
    tis_auc   = compute_auc(test_preds, test_labels, gene_cols, TIS_GENES, gene_symbol_to_idx)

    # Subtype-split evaluation
    luad_mask = np.array([r["subtype"] == "LUAD" for r in test_ds.records])
    lusc_mask = ~luad_mask
    subtype_results = {}
    for name, mask in [("LUAD", luad_mask), ("LUSC", lusc_mask)]:
        if mask.sum() == 0:
            continue
        p, l = test_preds[mask], test_labels[mask]
        subtype_results[name] = {
            "APM_PCC": compute_panel_pcc(p, l, gene_cols, APM_GENES, gene_symbol_to_idx),
            "TIS_PCC": compute_panel_pcc(p, l, gene_cols, TIS_GENES, gene_symbol_to_idx),
            "APM_AUC": compute_auc(p, l, gene_cols, APM_GENES, gene_symbol_to_idx),
            "TIS_AUC": compute_auc(p, l, gene_cols, TIS_GENES, gene_symbol_to_idx),
            "n":       int(mask.sum()),
        }

    fold_result = {
        "fold":         fold,
        "best_val_pcc": float(best_val_pcc),
        "APM_PCC":      float(apm_pcc),
        "TIS_PCC":      float(tis_pcc),
        "APM_AUC":      float(apm_auc),
        "TIS_AUC":      float(tis_auc),
        "gene_pccs":    {g.replace("_fpkm_uq", ""): float(gene_pccs[i])
                         for i, g in enumerate(gene_cols)},
        "subtype":      subtype_results,
    }
    fold_results.append(fold_result)

    print(f"\nFold {fold} Results:")
    print(f"  APM  PCC={apm_pcc:.4f}  AUC={apm_auc:.4f}")
    print(f"  TIS  PCC={tis_pcc:.4f}  AUC={tis_auc:.4f}")
    for name, res in subtype_results.items():
        print(f"  {name} (n={res['n']}): APM={res['APM_PCC']:.4f}, TIS={res['TIS_PCC']:.4f}")

    # Save checkpoint after this fold completes
    save_checkpoint(fold_results, fold)

print("\n All folds complete.")

2026-09-14 11:55:19,959 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 11:55:19,964 [INFO]   LUSC feature index: 478 unique patient barcodes


Folds:5, Patience: 3

FOLD 0 / 4


2026-09-14 11:55:20,502 [INFO] Dataset: 584 matched slides (LUAD: 301, LUSC: 283)
2026-09-14 11:55:20,506 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 11:55:20,509 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-09-14 11:55:20,627 [INFO] Dataset: 147 matched slides (LUAD: 75, LUSC: 72)
2026-09-14 11:55:20,630 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 11:55:20,636 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-09-14 11:55:20,791 [INFO] Dataset: 181 matched slides (LUAD: 77, LUSC: 104)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necess

Train: 584 | Val: 147 | Test: 181


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...


Fold 3 Results:
  APM  PCC=0.5080  AUC=0.7562
  TIS  PCC=0.5597  AUC=0.7921
  LUAD (n=77): APM=0.3230, TIS=0.4873
  LUSC (n=104): APM=0.5839, TIS=0.6008
Checkpoint saved after fold 3

FOLD 4 / 4


2026-09-14 13:22:38,249 [INFO] Dataset: 586 matched slides (LUAD: 303, LUSC: 283)
2026-09-14 13:22:38,255 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 13:22:38,258 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-09-14 13:22:38,409 [INFO] Dataset: 145 matched slides (LUAD: 73, LUSC: 72)
2026-09-14 13:22:38,415 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 13:22:38,418 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-09-14 13:22:38,573 [INFO] Dataset: 181 matched slides (LUAD: 77, LUSC: 104)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necess

Train: 586 | Val: 145 | Test: 181


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  Epoch   1 | Train loss=0.9802 PCC=0.0463 | Val loss=1.0969 PCC=0.1756
  Epoch   2 | Train loss=0.9601 PCC=0.2044 | Val loss=1.0630 PCC=0.3033
  Epoch   3 | Train loss=0.9063 PCC=0.3174 | Val loss=0.9972 PCC=0.3909
  Epoch   4 | Train loss=0.8152 PCC=0.4411 | Val loss=0.9118 PCC=0.4571
  Epoch   5 | Train loss=0.7349 PCC=0.5015 | Val loss=0.8641 PCC=0.4850
  Epoch   6 | Train loss=0.6685 PCC=0.5541 | Val loss=0.8312 PCC=0.5007
  Epoch   7 | Train loss=0.6387 PCC=0.5764 | Val loss=0.8224 PCC=0.5063
  Epoch   8 | Train loss=0.6128 PCC=0.5966 | Val loss=0.8312 PCC=0.5082
  Epoch   9 | Train loss=0.5983 PCC=0.6095 | Val loss=0.8188 PCC=0.5104
  Epoch  10 | Train loss=0.5666 PCC=0.6354 | Val loss=0.8100 PCC=0.5195
  Epoch  11 | Train loss=0.5565 PCC=0.6440 | Val loss=0.8242 PCC=0.5171
  Epoch  12 | Train loss=0.5329 PCC=0.6615 | Val loss=0.8065 PCC=0.5228
  Epoch  13 | Train loss=0.5194 PCC=0.6726 | Val loss=0.8259 PCC=0.5143
  Epoch  14 | Train loss=0.5061 PCC=0.6838 | Val loss=0.8188 PCC

In [10]:
# Cross-validation summary
print("CROSS-VALIDATION SUMMARY")

for metric in ["APM_PCC", "TIS_PCC", "APM_AUC", "TIS_AUC"]:
    vals = [r[metric] for r in fold_results]
    print(f"  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print()
for subtype in ["LUAD", "LUSC"]:
    print(f"  {subtype}")
    for metric in ["APM_PCC", "TIS_PCC", "APM_AUC", "TIS_AUC"]:
        vals = [r["subtype"][subtype][metric]
                for r in fold_results if subtype in r["subtype"]]
        if vals:
            print(f"  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

# Gene-level summary — top and bottom 5 across folds
print("\n Gene-level PCC (mean across folds)")
all_gene_symbols = list(fold_results[0]["gene_pccs"].keys())
mean_gene_pccs = {
    g: np.mean([r["gene_pccs"][g] for r in fold_results])
    for g in all_gene_symbols
}
sorted_genes = sorted(mean_gene_pccs.items(), key=lambda x: x[1], reverse=True)
print("  Top 5 genes:")
for g, r in sorted_genes[:5]:
    print(f"    {g:15s}: {r:.4f}")
print("  Bottom 5 genes:")
for g, r in sorted_genes[-5:]:
    print(f"    {g:15s}: {r:.4f}")

CROSS-VALIDATION SUMMARY
  APM_PCC     : 0.5008 ± 0.0088
  TIS_PCC     : 0.5809 ± 0.0155
  APM_AUC     : 0.7428 ± 0.0105
  TIS_AUC     : 0.8167 ± 0.0136

  LUAD
  APM_PCC     : 0.3150 ± 0.0138
  TIS_PCC     : 0.5075 ± 0.0264
  APM_AUC     : 0.7114 ± 0.0268
  TIS_AUC     : 0.8156 ± 0.0194
  LUSC
  APM_PCC     : 0.5837 ± 0.0145
  TIS_PCC     : 0.6264 ± 0.0241
  APM_AUC     : 0.7975 ± 0.0088
  TIS_AUC     : 0.8156 ± 0.0156

 Gene-level PCC (mean across folds)
  Top 5 genes:
    TIS            : 0.5776
    HLA-DRB1       : 0.5690
    NKG7           : 0.5454
    LAG3           : 0.5329
    HLA-DQA1       : 0.5294
  Bottom 5 genes:
    IDO1           : 0.3479
    PSMB5          : 0.3364
    CALR           : 0.2312
    PSMB6          : 0.2075
    ERAP2          : 0.1833


In [11]:
# Save final results JSON
import json

results_path = f"{OUTPUT_DIR}/results.json"
with open(results_path, "w") as f:
    json.dump(fold_results, f, indent=2, default=_json_default)

print(f"Results saved to: {results_path}")
print("\nModel weights saved:")
for pt in Path(OUTPUT_DIR).glob("*.pt"):
    print(f"  {pt.name}  ({pt.stat().st_size / 1e6:.1f} MB)")

Results saved to: /kaggle/working/results/results.json

Model weights saved:
  fold2_best_model.pt  (5.7 MB)
  fold0_best_model.pt  (5.7 MB)
  fold4_best_model.pt  (5.7 MB)
  fold1_best_model.pt  (5.7 MB)
  fold3_best_model.pt  (5.7 MB)


## Reproducibility

In [12]:
# Patient ID lists per split + leakage check
import os, json
import numpy as np
from sklearn.model_selection import KFold

EXPORT_DIR = f"{OUTPUT_DIR}/paper_artifacts"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Re-derive the exact per-fold train/val split
kf_check = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_splits = list(kf_check.split(df_dev))
assert len(fold_splits) == N_FOLDS

test_sids_set = set(df_test["submitter_id"])
split_record = {"test": sorted(test_sids_set)}

leakage_found = False
for fold, (train_idx, val_idx) in enumerate(fold_splits):
    train_sids = set(df_dev.iloc[train_idx]["submitter_id"])
    val_sids   = set(df_dev.iloc[val_idx]["submitter_id"])

    split_record[f"fold{fold}_train"] = sorted(train_sids)
    split_record[f"fold{fold}_val"]   = sorted(val_sids)

    # Leakage checks
    tt = train_sids & test_sids_set
    vt = val_sids & test_sids_set
    tv = train_sids & val_sids
    if tt or vt or tv:
        leakage_found = True
        print(f"Fold {fold}: train∩test={len(tt)} val∩test={len(vt)} train∩val={len(tv)}")

if not leakage_found:
    print("No patient-level overlap between train/val/test in any fold.")
else:
    print("LEAKAGE DETECTED — see above. Do not report results until resolved.")

with open(f"{EXPORT_DIR}/patient_id_splits.json", "w") as f:
    json.dump(split_record, f, indent=2)

print(f"Saved patient ID splits: {EXPORT_DIR}/patient_id_splits.json")
print(f"Test: {len(test_sids_set)} | "
      f"Fold0 train/val: {len(fold_splits[0][0])}/{len(fold_splits[0][1])}")

No patient-level overlap between train/val/test in any fold.
Saved patient ID splits: /kaggle/working/results/paper_artifacts/patient_id_splits.json
Test: 197 | Fold0 train/val: 632/158


In [14]:
# Raw predictions + labels per fold (reloaded from checkpoints)
import torch
from torch.utils.data import DataLoader
from pathlib import Path

# Fixed test set — identical across every fold by construction
test_ds = FiLMDataset(df_test, feature_dirs, gene_cols, clinical_cols,
                       n_tiles=None, deterministic=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

test_sids_ordered     = [r["sid"] for r in test_ds.records]      # row order guarantee
test_subtypes_ordered = [r["subtype"] for r in test_ds.records]  # (shuffle=False)

all_fold_preds = {}   # fold -> (n_test, n_genes) array
all_fold_labels = None

for fold in range(N_FOLDS):
    ckpt_path = Path(OUTPUT_DIR) / f"fold{fold}_best_model.pt"
    if not ckpt_path.exists():
        print(f"Fold {fold}: checkpoint not found, skipping (not completed yet?)")
        continue

    model = FiLMMILModel(feat_dim=1536, n_genes=len(gene_cols), use_film=FILM_ENABLED).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    loss_fn   = CompositeLoss()
    optimizer = torch.optim.Adam(model.parameters())  # unused (training=False), needed for signature

    _, _, test_preds, test_labels, test_panel_preds, test_panel_labels = run_epoch(model, test_loader, loss_fn, optimizer, 999, device, training=False, panel_idx=panel_idx, panel_gene_fallback_idx=panel_gene_fallback_idx,)
    all_fold_preds[fold] = test_preds
    all_fold_labels = test_labels  # identical every fold (same fixed test set)

    np.savez(
        f"{EXPORT_DIR}/fold{fold}_test_predictions.npz",
        preds=test_preds, labels=test_labels,
        submitter_id=np.array(test_sids_ordered),
        subtype=np.array(test_subtypes_ordered),
        gene_cols=np.array(gene_cols),
    )
    print(f"  Fold {fold}: saved raw predictions {test_preds.shape} "
          f"to fold{fold}_test_predictions.npz")

# Ensemble prediction (mean across folds, same fixed test set) — your
# headline number, since all folds evaluate the identical test set
ensemble_preds = np.mean(list(all_fold_preds.values()), axis=0)
np.savez(
    f"{EXPORT_DIR}/ensemble_test_predictions.npz",
    preds=ensemble_preds, labels=all_fold_labels,
    submitter_id=np.array(test_sids_ordered),
    subtype=np.array(test_subtypes_ordered),
    gene_cols=np.array(gene_cols),
)
print(f"Ensemble predictions saved ({len(all_fold_preds)} folds averaged)")

# Tidy long-format CSV for quick plotting (predicted vs actual, per gene, per patient)
import pandas as pd
rows = []
for i, sid in enumerate(test_sids_ordered):
    for g_idx, g in enumerate(gene_cols):
        rows.append({
            "submitter_id": sid,
            "subtype": test_subtypes_ordered[i],
            "gene": g.replace("_fpkm_uq", ""),
            "predicted": ensemble_preds[i, g_idx],
            "actual": all_fold_labels[i, g_idx],
        })
pd.DataFrame(rows).to_csv(f"{EXPORT_DIR}/ensemble_predictions_long.csv", index=False)
print(f"Long-format CSV for scatter plots saved ({len(rows)} rows)")

2026-09-14 13:53:21,752 [INFO]   LUAD feature index: 468 unique patient barcodes
2026-09-14 13:53:21,757 [INFO]   LUSC feature index: 478 unique patient barcodes
2026-09-14 13:53:21,923 [INFO] Dataset: 181 matched slides (LUAD: 77, LUSC: 104)


  Fold 0: saved raw predictions (181, 38) to fold0_test_predictions.npz


KeyboardInterrupt: 

In [ ]:
# Bootstrap CIs + permutation p-values
N_BOOT = 2000
N_PERM = 2000
rng = np.random.default_rng(0)

def bootstrap_ci(preds, labels, gene_list, n_boot=N_BOOT):
    n = preds.shape[0]
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)  # resample patients with replacement
        vals.append(compute_panel_pcc(preds[idx], labels[idx], gene_cols, gene_list, gene_symbol_to_idx))
    vals = np.array(vals)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def permutation_pvalue(preds, labels, gene_list, n_perm=N_PERM):
    observed = compute_panel_pcc(preds, labels, gene_cols, gene_list, gene_symbol_to_idx)
    n = preds.shape[0]
    null_vals = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        null_vals[i] = compute_panel_pcc(preds, labels[perm_idx], gene_cols, gene_list, gene_symbol_to_idx)
    p = (np.sum(null_vals >= observed) + 1) / (n_perm + 1)
    return float(observed), float(p)

stats_summary = {}
targets = {"APM": APM_GENES, "TIS": TIS_GENES}

# Per-fold + ensemble
sources = {**{f"fold{f}": p for f, p in all_fold_preds.items()}, "ensemble": ensemble_preds}
for name, preds in sources.items():
    stats_summary[name] = {}
    for panel, genes in targets.items():
        mean_pcc, lo, hi = bootstrap_ci(preds, all_fold_labels, genes)
        observed, p = permutation_pvalue(preds, all_fold_labels, genes)
        stats_summary[name][panel] = {
            "PCC": observed, "bootstrap_mean": mean_pcc,
            "CI95_low": lo, "CI95_high": hi, "perm_p": p,
        }
        print(f"{name:10s} {panel}: PCC={observed:.4f} 95%CI=[{lo:.4f},{hi:.4f}] p={p:.4f}")

with open(f"{EXPORT_DIR}/bootstrap_permutation_stats.json", "w") as f:
    json.dump(stats_summary, f, indent=2)
print(f"Saved: {EXPORT_DIR}/bootstrap_permutation_stats.json")

In [ ]:
# y_means/y_stds, gene order, model & training config
from train_film_mil import SUBTYPE_MAP

config = {
    "gene_cols": list(gene_cols),                 # exact order model expects/outputs
    "clinical_cols": list(clinical_cols),
    "y_means": {g: float(y_means[g]) for g in gene_cols} if hasattr(y_means, "__getitem__") else list(map(float, y_means)),
    "y_stds":  {g: float(y_stds[g])  for g in gene_cols} if hasattr(y_stds, "__getitem__") else list(map(float, y_stds)),
    "SUBTYPE_MAP": SUBTYPE_MAP,
    "APM_GENES": APM_GENES,
    "TIS_GENES": TIS_GENES,
    "model": {
        "architecture": "FiLMMILModel",
        "feat_dim": 1536,
        "n_genes": len(gene_cols),
    },
    "training": {
        "n_folds": N_FOLDS,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "use_film": FILM_ENABLED,
        "optimizer": "Adam", "lr": 1e-4, "weight_decay": 1e-5,
        "scheduler": "ReduceLROnPlateau", "scheduler_patience": 5, "scheduler_factor": 0.5,
        "batch_size": 1,
        "loss": "CompositeLoss (per-sample MSE + batch-wise composite every 16 slides)",
        "gradient_clip_norm": 1.0,
        "dev_test_split_seed": 42, "kfold_random_state": 42,
    },
    "feature_extractor": {
        "name": "UNI2-h Pretrained vision backbone (ViT-H/14 via DINOv2)",                    # ⚠ FILL IN: exact checkpoint/version/revision you used
        "patch_size_px": "256 x 256",
        "magnification": "20x",
    },
}

with open(f"{EXPORT_DIR}/model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved model_config.json")

In [ ]:
# Cohort characteristics table
candidate_cols = {
    "age": ["age_years"],
    "sex": ["demographic.gender"],
    "stage": ["diagnoses.0.ajcc_pathologic_stage"],
}
found_cols = {}
for label, candidates in candidate_cols.items():
    for c in candidates:
        if c in df.columns:
            found_cols[label] = c
            break

print("Columns found for Table 1:", found_cols)
missing = [k for k in candidate_cols if k not in found_cols]
if missing:
    print(f"Not found in metadata CSV: {missing}")

splits = {"Test": df_test, "Dev (train+val)": df_dev, "Full cohort": df}
table1_rows = []
for split_name, split_df in splits.items():
    for subtype in ["LUAD", "LUSC"]:
        sub = split_df[split_df["cancer_type"] == subtype] if "cancer_type" in split_df.columns else split_df
        row = {"split": split_name, "subtype": subtype, "n": len(sub)}
        if "age" in found_cols:
            row["age_mean"] = float(sub[found_cols["age"]].mean())
            row["age_sd"]   = float(sub[found_cols["age"]].std())
        if "sex" in found_cols:
            row["sex_counts"] = sub[found_cols["sex"]].value_counts().to_dict()
        if "stage" in found_cols:
            row["stage_counts"] = sub[found_cols["stage"]].value_counts().to_dict()
        table1_rows.append(row)

table1_df = pd.DataFrame(table1_rows)
table1_df.to_csv(f"{EXPORT_DIR}/table1_cohort_characteristics.csv", index=False)
print(table1_df)
print(f"Saved: {EXPORT_DIR}/table1_cohort_characteristics.csv")